# Hierarchical ReLU-LoRA Spawning — OLMoE Final Notebook

**Adapted from hrlora-final-results.ipynb for OLMoE-1B-7B**

**Execution order (run top-to-bottom):**
1. Install & model load
2. All class definitions (HierarchicalExpert, SaturationMonitor, HierarchicalOLMoEExperts)
3. Apply wrapper to model
4. Training loop
5. HumanEval evaluation

**Model:** allenai/OLMoE-1B-7B-0924
- 16 layers, 64 experts/layer, top-8 routing
- model_dim = 2048, ffn_dim = 1024
- Experts are PACKED tensors (like Phi-MoE)
- VRAM: ~7GB (fits easily on Kaggle T4)

## Section 1 — Install & Model Load

In [ ]:
# Install (run once per session)
!pip install transformers>=4.41.0 accelerate datasets peft torch>=2.2.0 tabulate -q

In [ ]:
# Load OLMoE model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gc
import os

# Clear any lingering VRAM
torch.cuda.empty_cache()
gc.collect()

MODEL_ID = "allenai/OLMoE-1B-7B-0924"

# Save directory setup
if os.path.exists('/kaggle'):
    SAVE_ROOT = "/kaggle/working/hrlora_checkpoints"
    print("Detected: Kaggle")
elif os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_ROOT = "/content/drive/MyDrive/hrlora_checkpoints"
    except:
        SAVE_ROOT = "/content/hrlora_checkpoints"
    print("Detected: Colab")
else:
    SAVE_ROOT = "./hrlora_checkpoints"
    print("Detected: Local")

os.makedirs(SAVE_ROOT, exist_ok=True)
print(f"Checkpoints: {SAVE_ROOT}")

print(f"\nLoading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_olmoe = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
model_olmoe.config.use_cache = False  # Required for training

print(f"\nModel loaded!")
print(f"  Layers: {model_olmoe.config.num_hidden_layers}")
print(f"  Experts: {model_olmoe.config.num_experts}")
print(f"  Top-k: {model_olmoe.config.num_experts_per_tok}")
print(f"  model_dim: {model_olmoe.config.hidden_size}")
print(f"  ffn_dim: {model_olmoe.config.intermediate_size}")
print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Section 2 — Class Definitions

In [ ]:
import torch
import torch.nn as nn

class HierarchicalExpert(nn.Module):
    """
    Low-rank LoRA sub-adapter.  Computes: L(x) = (x A^T B^T) * scaling
    
    Key fix: B is zero-initialized by default. This guarantees that at the
    moment of spawn, the sub-adapter's contribution is exactly zero, which
    is what makes Check 2 (zero-loss-spike) provably pass.
    """
    def __init__(self, in_features, out_features, base_rank=16, lora_alpha=32):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.rank    = base_rank
        self.scaling = lora_alpha / base_rank

        # Standard LoRA init: A ~ N(0, 0.01), B = 0
        # B=0 means output is zero at init → no loss spike on insertion
        self.A = nn.Parameter(torch.randn(base_rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, base_rank))   # FIX: was randn

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Cast A and B to match x's dtype at runtime.
        A = self.A.to(x.dtype)
        B = self.B.to(x.dtype)
        return (x @ A.t() @ B.t()) * self.scaling

print("HierarchicalExpert defined.")

In [ ]:
import torch
from collections import deque

class SaturationMonitor:
    """
    Bivariate saturation trigger (§3.2 of proposal).

    Fires when BOTH conditions hold over a rolling window of W steps:
      1. Learning Plateau : |d/dt g_k| < tau_plateau
         where g_k = mean(||B[:,r]|| * ||A[r,:]||) is the rank importance score
      2. High Residual Error: L_k > alpha * L_batch
         i.e. expert's error exceeds the batch mean by factor alpha
    """

    def __init__(self, expert_id: int, beta: float = 0.9,
                 window: int = 50, tau_plateau: float = 1e-4,
                 alpha: float = 1.25):
        self.expert_id   = expert_id
        self.beta        = beta
        self.window      = window
        self.tau_plateau = tau_plateau
        self.alpha       = alpha

        # EMA of rank importance
        self.g_ema      = 0.0
        self.g_ema_prev = 0.0

        # Rolling window hits
        self.plateau_hits = deque(maxlen=window)
        self.error_hits   = deque(maxlen=window)

        # Logging
        self.history_importance = []
        self.history_ratio      = []
        self.has_spawned        = False   # one-shot guard

    def _rank_importance(self, A: torch.Tensor, B: torch.Tensor) -> float:
        """||B[:,r]|| * ||A[r,:]|| averaged over ranks."""
        with torch.no_grad():
            A_norms = A.float().norm(dim=1)        # (rank,)
            B_norms = B.float().norm(dim=0)        # (rank,)
            return (A_norms * B_norms).mean().item()

    def update(self, A: torch.Tensor, B: torch.Tensor,
               expert_loss: float, batch_loss: float) -> bool:
        """
        Call once per training step.
        Returns True when spawn trigger fires.
        """
        if self.has_spawned:
            return False

        # ── Rank importance EMA ──────────────────────────────────────────
        g_now = self._rank_importance(A, B)
        self.g_ema_prev = self.g_ema
        self.g_ema      = self.beta * self.g_ema + (1 - self.beta) * g_now

        # ── Condition 1: plateau ─────────────────────────────────────────
        delta = abs(self.g_ema - self.g_ema_prev)
        plateau_hit = (delta < self.tau_plateau)
        self.plateau_hits.append(plateau_hit)

        # ── Condition 2: high residual error ─────────────────────────────
        ratio = expert_loss / (batch_loss + 1e-8)
        error_hit = (ratio > self.alpha)
        self.error_hits.append(error_hit)

        # ── Logging ──────────────────────────────────────────────────────
        self.history_importance.append(self.g_ema)
        self.history_ratio.append(ratio)

        # ── Trigger check ────────────────────────────────────────────────
        if (len(self.plateau_hits) == self.window and
            all(self.plateau_hits) and all(self.error_hits)):
            return True
        return False

    def reset_after_spawn(self):
        self.plateau_hits.clear()
        self.error_hits.clear()
        self.has_spawned = True

print("SaturationMonitor defined.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HierarchicalOLMoEExperts(nn.Module):
    """
    Drop-in replacement for OLMoE's packed expert block (OlmoeExperts).

    OLMoE's SparseMoeBlock calls: self.experts(hidden_states, router_top_k_indices, router_top_k_weights)
    where hidden_states is [batch*seq, model_dim]. We intercept this, compute the
    original output, and add per-expert LoRA corrections:

        E_k(x) = W_k x + L_{k,0}(x) + Σ_j ReLU(w_{k,j}^T x) L_{k,j}(x)

    NOTE on dimensions
    ------------------
    OLMoE packed weights:
      gate_up_proj.shape = [num_experts, hidden_size, intermediate_size*2]
      down_proj.shape    = [num_experts, hidden_size, intermediate_size]
    
    hidden_states arriving at the experts is [T, hidden_size].
    Our LoRA correction lives in hidden_size → hidden_size space,
    i.e. lora_in = lora_out = hidden_size = model_dim.
    """

    def __init__(self, original_experts, base_rank: int = 16, lora_alpha: int = 32):
        super().__init__()
        self.original = original_experts

        # Freeze every original expert parameter
        for p in self.original.parameters():
            p.requires_grad = False

        # Infer packed-weight dimensions from OLMoE structure
        # down_proj: [num_experts, hidden_size, intermediate_size]
        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype

        # LoRA correction lives in hidden_size space (input to & output from experts)
        self.lora_dim = self.out_f   # = hidden_size = model_dim

        # Base LoRA per expert — registered as nn.ModuleList so optimizer finds them
        dev   = self.original.down_proj.device
        dtype = self.original.down_proj.dtype

        self.base_loras = nn.ModuleList([
            HierarchicalExpert(
                in_features=self.lora_dim,
                out_features=self.lora_dim,
                base_rank=base_rank,
                lora_alpha=lora_alpha,
            ).to(dev).to(dtype)
            for _ in range(self.num_experts)
        ])

        # Spawned adapters: plain Python lists, added to optimizer manually on spawn.
        # spawn_loras[k] = [HierarchicalExpert, ...]
        # spawn_gates[k] = [nn.Parameter of shape [lora_dim], ...]
        self.spawn_loras: list[list] = [[] for _ in range(self.num_experts)]
        self.spawn_gates: list[list] = [[] for _ in range(self.num_experts)]

    @property
    def device(self) -> torch.device:
        return self.original.down_proj.device

    def spawn(self, expert_id: int, rank: int = 8,
              weight_grad: torch.Tensor | None = None) -> list:
        """
        Spawn a new ReLU-gated LoRA sub-adapter inside expert `expert_id`.

        Initialization protocol
        -----------------------
        • A  ← top-r rows of V^H from SVD of grad (residual-gradient init, LoRA-GA style)
                Falls back to small random if grad is None or SVD fails.
        • B  ← zero   (guarantees exact zero output at spawn → no loss spike, Check 2)
        • w  ← N(0, σ²) where σ = 1e-3 · Var(W_k)   (per proposal §3.3)

        Returns list of new nn.Parameters to add to the optimizer.
        """
        dev, dtype = self.device, self.dtype

        lora = HierarchicalExpert(
            in_features=self.lora_dim,
            out_features=self.lora_dim,
            base_rank=rank,
            lora_alpha=2 * rank,
        ).to(dev).to(dtype)

        # Gradient-informed A init, B always stays zero
        if weight_grad is not None:
            try:
                # weight_grad: [lora_dim, lora_dim]
                U, S, Vh = torch.linalg.svd(weight_grad.float(), full_matrices=False)
                with torch.no_grad():
                    lora.A.copy_(Vh[:rank].to(dtype))   # top-r rows of V^H
                    # lora.B stays zero → output = 0 at spawn (Check 2 guaranteed)
                print(f"  [spawn] Expert {expert_id}: SVD success, top singular = {S[0].item():.4f}")
            except Exception as e:
                print(f"  [spawn] Expert {expert_id}: SVD failed ({e}), using default random A init.")

        # Symmetry-breaking gate: σ = c · Var(W_k)  (proposal eq. 3.3)
        sigma = 1e-3 * self.original.down_proj[expert_id].float().var().item()
        gate = nn.Parameter(
            torch.randn(self.lora_dim, device=dev, dtype=dtype) * sigma
        )

        self.spawn_loras[expert_id].append(lora)
        self.spawn_gates[expert_id].append(gate)

        # Return ALL new leaf parameters so the caller can add them to the optimizer
        return list(lora.parameters()) + [gate]

    def forward(self,
                hidden_states: torch.Tensor,   # [T, model_dim]
                router_top_k_indices: torch.Tensor,   # [T, top_k]
                router_top_k_weights: torch.Tensor,   # [T, top_k]
                ) -> torch.Tensor:
        """
        Forward pass matching OLMoE's OlmoeExperts.forward() signature.
        """
        # Call original experts
        orig_out = self.original.forward(hidden_states, router_top_k_indices, router_top_k_weights)
        
        # Initialize correction tensor
        correction = torch.zeros_like(orig_out)

        T, top_k = router_top_k_indices.shape
        
        for k in range(self.num_experts):
            # Find which tokens route to expert k (in any top-k slot)
            mask = (router_top_k_indices == k)  # [T, top_k] bool
            if not mask.any():
                continue

            # Effective routing weight for expert k per token
            # Sum weights across all top-k slots where this expert was selected
            eff_w = (router_top_k_weights * mask.float()).sum(dim=1)  # [T]
            
            # Only process tokens that actually route to this expert
            token_mask = eff_w > 0  # [T] bool
            if not token_mask.any():
                continue

            # Base LoRA correction for all tokens (sparse apply via eff_w)
            base_out = self.base_loras[k](hidden_states)  # [T, lora_dim]
            correction = correction + base_out * eff_w.unsqueeze(-1)

            # Spawned sub-adapter corrections
            for gate_vec, sub_lora in zip(self.spawn_gates[k], self.spawn_loras[k]):
                # ReLU scalar gate per token (proposal §3.1)
                g = F.relu(hidden_states @ gate_vec)  # [T]
                sub_out = sub_lora(hidden_states)     # [T, lora_dim]
                correction = correction + sub_out * (g * eff_w).unsqueeze(-1)

        return (orig_out + correction).to(hidden_states.dtype)

print("HierarchicalOLMoEExperts defined.")

## Section 3 — Apply Wrapper to Model

In [ ]:
# Freeze all base model parameters
for param in model_olmoe.parameters():
    param.requires_grad = False
print("All base parameters frozen.")

# Configuration
TARGET_LAYERS = [8, 12, 15]  # Layers to apply hierarchical LoRA
BASE_RANK = 16
LORA_ALPHA = 32

# Store wrappers for training
wrappers = {}

for layer_idx in TARGET_LAYERS:
    layer = model_olmoe.model.layers[layer_idx]
    moe_block = layer.mlp
    original_experts = moe_block.experts
    
    # Create wrapper
    wrapper = HierarchicalOLMoEExperts(
        original_experts=original_experts,
        base_rank=BASE_RANK,
        lora_alpha=LORA_ALPHA,
    )
    
    # Replace the experts module
    moe_block.experts = wrapper
    wrappers[layer_idx] = wrapper
    
    trainable = sum(p.numel() for p in wrapper.parameters() if p.requires_grad)
    print(f"Layer {layer_idx}: Wrapped with {trainable:,} trainable params")

total_trainable = sum(p.numel() for p in model_olmoe.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_trainable:,}")
print(f"VRAM after wrapping: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Section 4 — Data Loader

In [ ]:
from datasets import load_dataset

print("Loading CodeSearchNet Python dataset...")
dataset = load_dataset("code_search_net", "python", split="train", trust_remote_code=True)

# Filter for reasonable length functions
dataset = dataset.filter(lambda x: 50 < len(x['func_code_string']) < 1000)
dataset = dataset.shuffle(seed=42).select(range(min(10000, len(dataset))))

print(f"Dataset size: {len(dataset)} examples")
print(f"\nSample:\n{dataset[0]['func_code_string'][:300]}...")

## Section 5 — Training Loop

In [ ]:
# ── Training Hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE = 2
MAX_LENGTH = 256
LR = 2e-5
N_STEPS = 1000
SAVE_EVERY = 200
LOG_EVERY = 50

# Saturation monitor settings
WINDOW = 50
TAU_PLATEAU = 1e-4
ALPHA = 1.15

# Collect trainable parameters
trainable_params = []
for wrapper in wrappers.values():
    trainable_params.extend([p for p in wrapper.parameters() if p.requires_grad])

optimizer = torch.optim.AdamW(trainable_params, lr=LR)

# Create monitors for subset of experts (monitoring all 64 is expensive)
monitors = {}
MONITORED_EXPERTS = [0, 8, 16, 32]  # Sample of experts to monitor
for layer_idx, wrapper in wrappers.items():
    monitors[layer_idx] = {}
    for expert_id in MONITORED_EXPERTS:
        monitors[layer_idx][expert_id] = SaturationMonitor(
            expert_id=expert_id,
            window=WINDOW,
            tau_plateau=TAU_PLATEAU,
            alpha=ALPHA
        )

print(f"Optimizer: AdamW, lr={LR}")
print(f"Trainable params: {sum(p.numel() for p in trainable_params):,}")
print(f"Monitoring experts: {MONITORED_EXPERTS} across layers {TARGET_LAYERS}")

In [ ]:
# ── Training Loop ────────────────────────────────────────────────────────────
model_olmoe.train()
loss_log = []
spawn_events = []

print(f"\nStarting training for {N_STEPS} steps...")
print("-" * 60)

for step in range(N_STEPS):
    # Sample batch
    indices = torch.randint(0, len(dataset), (BATCH_SIZE,)).tolist()
    texts = [dataset[i]['func_code_string'] for i in indices]
    
    encodings = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt'
    ).to(model_olmoe.device)
    
    # Create labels (shift by 1 for causal LM)
    labels = encodings['input_ids'].clone()
    labels[:, :-1] = encodings['input_ids'][:, 1:]
    labels[:, -1] = -100
    
    optimizer.zero_grad()
    
    outputs = model_olmoe(
        input_ids=encodings['input_ids'],
        attention_mask=encodings['attention_mask'],
        labels=labels
    )
    
    loss = outputs.loss
    loss_log.append(loss.item())
    
    loss.backward()
    
    # Check saturation monitors and spawn if needed
    for layer_idx, wrapper in wrappers.items():
        for expert_id, monitor in monitors[layer_idx].items():
            base_lora = wrapper.base_loras[expert_id]
            
            # Expert loss proxy (simplified - in full impl track per-expert loss)
            expert_loss_proxy = loss.item() * (1.0 + 0.05 * (expert_id % 8 - 3.5))
            
            trigger = monitor.update(
                A=base_lora.A.data,
                B=base_lora.B.data,
                expert_loss=expert_loss_proxy,
                batch_loss=loss.item()
            )
            
            if trigger:
                print(f"\n[Step {step}] SPAWN: Layer {layer_idx}, Expert {expert_id}")
                
                # Compute gradient for SVD init from base LoRA gradients
                if base_lora.A.grad is not None and base_lora.B.grad is not None:
                    weight_grad = base_lora.B.grad.float() @ base_lora.A.grad.float()
                else:
                    weight_grad = None
                
                new_params = wrapper.spawn(expert_id, rank=8, weight_grad=weight_grad)
                optimizer.add_param_group({'params': new_params, 'lr': LR})
                monitor.reset_after_spawn()
                spawn_events.append((step, layer_idx, expert_id))
    
    optimizer.step()
    
    # Logging
    if step % LOG_EVERY == 0:
        vram = torch.cuda.memory_allocated() / 1e9
        total_subs = sum(sum(len(sl) for sl in w.spawn_loras) for w in wrappers.values())
        print(f"Step {step:4d} | Loss: {loss.item():.4f} | VRAM: {vram:.1f}GB | Spawned: {total_subs}")
    
    # Save checkpoint
    if step > 0 and step % SAVE_EVERY == 0:
        ckpt_path = os.path.join(SAVE_ROOT, f"olmoe_hrlora_step{step}.pt")
        state = {
            'step': step,
            'wrappers': {k: v.state_dict() for k, v in wrappers.items()},
            'optimizer': optimizer.state_dict(),
            'loss_log': loss_log,
            'spawn_events': spawn_events,
        }
        torch.save(state, ckpt_path)
        print(f"  Saved: {ckpt_path}")

# Final save
final_path = os.path.join(SAVE_ROOT, "olmoe_hrlora_final.pt")
torch.save({
    'step': N_STEPS,
    'wrappers': {k: v.state_dict() for k, v in wrappers.items()},
    'loss_log': loss_log,
    'spawn_events': spawn_events,
}, final_path)
print(f"\nTraining complete! Final: {final_path}")
print(f"Total spawn events: {len(spawn_events)}")

In [ ]:
# ── Plot Training Progress ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(loss_log, alpha=0.7)
for step, layer, expert in spawn_events:
    axes[0].axvline(x=step, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss (red = spawn events)')
axes[0].grid(alpha=0.3)

# Smoothed loss
window = 50
if len(loss_log) > window:
    smoothed = [sum(loss_log[max(0,i-window):i+1])/min(i+1, window) for i in range(len(loss_log))]
    axes[1].plot(smoothed)
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Smoothed Loss')
    axes[1].set_title(f'Smoothed Loss (window={window})')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_ROOT, 'training_curves.png'), dpi=150)
plt.show()

## Section 6 — HumanEval Evaluation

In [ ]:
def generate_completion(prompt, max_new_tokens=256):
    """Generate code completion for HumanEval."""
    model_olmoe.eval()
    
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model_olmoe.device)
    
    with torch.no_grad():
        outputs = model_olmoe.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
    new_text = completion[len(prompt):]
    
    # Stop at common end markers
    for stop in ['\ndef ', '\nclass ', '\n# ', '\nif __name__']:
        if stop in new_text:
            new_text = new_text[:new_text.index(stop)]
    
    return new_text

# Quick test
test_prompt = '''def add(a, b):
    """Add two numbers."""
    return'''

print("Test prompt:")
print(test_prompt)
print("\nGenerated:")
print(generate_completion(test_prompt, max_new_tokens=30))

In [ ]:
# ── HumanEval Benchmark ──────────────────────────────────────────────────────
import json

try:
    from human_eval.data import read_problems
    from human_eval.evaluation import evaluate_functional_correctness
    
    problems = read_problems()
    print(f"Loaded {len(problems)} HumanEval problems")
    
    samples = []
    for i, (task_id, problem) in enumerate(problems.items()):
        completion = generate_completion(problem['prompt'], max_new_tokens=256)
        samples.append({'task_id': task_id, 'completion': completion})
        if (i + 1) % 20 == 0:
            print(f"Generated {i+1}/{len(problems)}")
    
    samples_path = os.path.join(SAVE_ROOT, 'humaneval_samples.jsonl')
    with open(samples_path, 'w') as f:
        for s in samples:
            f.write(json.dumps(s) + '\n')
    
    results = evaluate_functional_correctness(samples_path)
    
    print(f"\n{'='*50}")
    print(f"HUMANEVAL RESULTS")
    print(f"{'='*50}")
    print(f"Pass@1: {results['pass@1']*100:.2f}%")
    
except ImportError:
    print("human_eval not installed. Running simplified test...")
    
    test_cases = [
        ("def add(a, b):\n    return", "a + b"),
        ("def is_even(n):\n    return", "n % 2 == 0"),
        ("def reverse_string(s):\n    return", "s[::-1]"),
    ]
    
    correct = 0
    for prompt, expected in test_cases:
        completion = generate_completion(prompt, max_new_tokens=20)
        key_op = expected.split()[0]
        if key_op in completion:
            correct += 1
            print(f"PASS: {prompt.split('def ')[1].split('(')[0]}")
        else:
            print(f"FAIL: {prompt.split('def ')[1].split('(')[0]} -> {completion[:30]}")
    
    print(f"\nSimplified: {correct}/{len(test_cases)}")

In [ ]:
# ── Save Results Summary ─────────────────────────────────────────────────────
summary = {
    'model': MODEL_ID,
    'total_steps': N_STEPS,
    'final_loss': loss_log[-1] if loss_log else None,
    'spawn_events': len(spawn_events),
    'target_layers': TARGET_LAYERS,
    'config': {
        'base_rank': BASE_RANK,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'window': WINDOW,
        'tau_plateau': TAU_PLATEAU,
        'alpha': ALPHA,
    }
}

summary_path = os.path.join(SAVE_ROOT, 'experiment_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print("Experiment Summary")
print("="*50)
for k, v in summary.items():
    print(f"{k}: {v}")
print(f"\nSaved: {summary_path}")